# Phase B - CellChat / LIANA (communication features)

Communication-Aware ML for MDD. This notebook runs on Colab (CPU, High-RAM)
and reads the Phase A checkpoint from Drive.

**Step 0** below is a read-only diagnostic: per-donor x cell-type nucleus
counts, used to decide granularity (broad vs hybrid) *before* we build any
features.

### Dependencies

In [ ]:
# scanpy/liana/cell2cell are not preinstalled on Colab. Install together so
# the resolver stays consistent; you may need one Runtime > Restart after.
!pip install -q scanpy pyarrow liana cell2cell

### Boot cell

Mounts Drive, pulls the latest repo (so `src/functions.py` is current), and
puts `src/` on the path. `%autoreload 2` means no kernel restart after a pull.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT   = '/content/drive/MyDrive/MLCB_team_project'
RAW_DIR        = os.path.join(PROJECT_ROOT, 'data', 'raw')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'data', 'checkpoints')
for d in (RAW_DIR, CHECKPOINT_DIR):
    os.makedirs(d, exist_ok=True)

REPO_URL = 'https://github.com/andreas824/MLCB_team_project.git'
REPO_DIR = '/content/MLCB_team_project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

# autoreload is optional: Colab's Python 3.12 + IPython 7.34 ship a broken
# autoreload (uses the removed `imp` module), so never let it break boot.
try:
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
except Exception as e:
    print(f'autoreload unavailable ({e}); continuing. Restart runtime after a pull if functions.py changes.')

from functions import build_condition_map, load_dataset  # noqa: F401

### Step 0 - per-donor x cell-type count diagnostic

Read-only sanity check. Decides granularity (broad vs hybrid) before we build
any features. We only need `.obs`, so we open the checkpoint in backed mode and
never load the 5.4 GB expression matrix into RAM.

In [ ]:
# === Phase B - Step 0: locate checkpoint, then per-donor x cell-type counts ===
import os, glob
import numpy as np
import pandas as pd
import scanpy as sc

CHECKPOINT_DIR = globals().get(
    'CHECKPOINT_DIR',
    '/content/drive/MyDrive/MLCB_team_project/data/checkpoints',
)

# --- locate the Phase A checkpoint, tolerant of name/location drift ----------
patterns = [
    os.path.join(CHECKPOINT_DIR, 'phaseA_final_for_cellchat.h5ad'),
    os.path.join(CHECKPOINT_DIR, 'phaseA*cellchat*.h5ad'),
    '/content/drive/MyDrive/**/phaseA*cellchat*.h5ad',
]
cands = []
for pat in patterns:
    cands = sorted(glob.glob(pat, recursive=True))
    if cands:
        break

if not cands:
    print("Checkpoint not found. What IS on your Drive:\n")
    print("MyDrive top-level:")
    for n in sorted(os.listdir('/content/drive/MyDrive')):
        print("   ", n)
    pr = '/content/drive/MyDrive/MLCB_team_project'
    if os.path.isdir(pr):
        print(f"\nFiles under {pr}:")
        for root, _, files in os.walk(pr):
            for f in files:
                print("   ", os.path.join(root, f))
    else:
        print("\n(no MLCB_team_project in MyDrive -- the data may be in")
        print(" 'Shared with me'. In Drive, right-click the shared folder ->")
        print(" Organise -> Add shortcut -> My Drive, then re-run.)")
    raise FileNotFoundError("checkpoint not found; see listing above")

CKPT = cands[0]
print(f"Using checkpoint: {CKPT}  ({os.path.getsize(CKPT)/1e9:.2f} GB)")
for c in cands[1:]:
    print("  (also found, ignored):", c)

adata = sc.read_h5ad(CKPT, backed='r')   # backed='r' => obs only, no matrix in RAM
obs = adata.obs.copy()

# cache obs as a small parquet so future diagnostics skip the big file
obs.to_parquet(os.path.join(CHECKPOINT_DIR, 'phaseA_obs.parquet'))

# ---- 0. dataset sanity: 71 donors, sex x condition layout -------------------
donor_meta = obs.drop_duplicates('donor_id').set_index('donor_id')[['sex', 'condition']]
print(f"\ndonors: {donor_meta.shape[0]}   nuclei: {obs.shape[0]:,}")
print("\nsex x condition (donors):")
print(pd.crosstab(donor_meta['sex'], donor_meta['condition'], margins=True))

# ---- helpers ----------------------------------------------------------------
THRESHOLDS = [5, 10, 20, 30, 50]
N_DONORS = donor_meta.shape[0]

def per_donor_table(col):
    return pd.crosstab(obs['donor_id'], obs[col])

def summarize(counts, label):
    print(f"\n{'='*70}\n{label}: per-donor count distribution\n{'='*70}")
    desc = counts.describe(percentiles=[.1, .25, .5, .75, .9]).T
    desc = desc[['min', '10%', '50%', '90%', 'max']].round(0).astype(int)
    desc.insert(0, 'donors_with_zero', (counts == 0).sum().values)
    print(desc.to_string())
    print(f"\n# donors (of {N_DONORS}) with >= threshold cells of each type:")
    print(pd.DataFrame({f">= {t}": (counts >= t).sum() for t in THRESHOLDS}).to_string())

# ---- 1. BROAD granularity ---------------------------------------------------
broad = per_donor_table('cell_type_broad')
if 'Mix' in broad.columns:
    broad = broad.drop(columns='Mix')
summarize(broad, "BROAD cell types (cell_type_broad)")

# ---- 2. FINE clusters the hypothesis names ----------------------------------
fine = per_donor_table('cell_type_fine')
PAIR = ['Mic1', 'ExN10_L46']
missing = [c for c in PAIR if c not in fine.columns]
if missing:
    print(f"\n!! WARNING: expected fine clusters not found: {missing}")
PAIR = [c for c in PAIR if c in fine.columns]
summarize(fine[PAIR], "HYPOTHESIS-PAIR fine clusters")

# ---- 3. fine cluster as a fraction of its broad parent ----------------------
parent = {'Mic1': 'Mic', 'ExN10_L46': 'ExN'}
print(f"\n{'='*70}\nfine-vs-parent share (median per-donor % of parent broad type)\n{'='*70}")
for fine_cl in PAIR:
    p = parent[fine_cl]
    if p in broad.columns:
        frac = (fine[fine_cl] / broad[p].replace(0, np.nan) * 100)
        print(f"  {fine_cl:11s} as % of {p:3s}: "
              f"median {frac.median():.1f}%  (range {frac.min():.1f}-{frac.max():.1f}%)")

# ---- 4. decision helper -----------------------------------------------------
print(f"\n{'='*70}\nGRANULARITY VERDICT HELPER\n{'='*70}")
for fine_cl in PAIR:
    n10 = int((fine[fine_cl] >= 10).sum())
    n20 = int((fine[fine_cl] >= 20).sum())
    print(f"  {fine_cl:11s}: {n10}/{N_DONORS} donors >=10 cells, {n20}/{N_DONORS} >=20 cells")
print("\nRule of thumb: BOTH clusters clearing ~10-20 cells in MOST donors -> HYBRID feasible.")
print("Many donors short -> stay BROAD.")

### Step 1 - Hybrid label + QC

Build the hybrid communication label (`compute_cc_label` in `functions.py`):
broad classes, with `ExN10_L46` split out of `ExN` (the remainder become
`ExN_other`); `Mic` is already exactly `Mic1`, so it needs no split. Then
sanity-check per-donor counts on the 8-node label, and check whether microglia
sparsity is confounded with diagnosis or sex - a leakage path through
missingness if MDD donors systematically have fewer microglia.

**Re-run the boot cell first** so the freshly pushed `functions.py` is pulled;
the cell below force-reloads the module (Colab's autoreload is unavailable).

In [ ]:
# === Phase B - Step 1 QC: hybrid label counts + microglia/diagnosis confound ===
# Re-run the BOOT cell first (it git-pulls functions.py), then run this.
import importlib, functions
importlib.reload(functions)                 # pick up freshly pulled functions.py
from functions import compute_cc_label

import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

CHECKPOINT_DIR = globals().get('CHECKPOINT_DIR',
    '/content/drive/MyDrive/MLCB_team_project/data/checkpoints')

# obs cached by Step 0 -> no need to touch the 5.8 GB matrix
obs = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseA_obs.parquet'))
obs['cc_label'] = compute_cc_label(obs['cell_type_broad'], obs['cell_type_fine'])

cc = obs[obs['cc_label'] != 'Mix'].copy()           # drop Mix for the CCC graph
print("cc_label nodes:", sorted(cc['cc_label'].unique()))

# ---- per-donor counts on the hybrid label -----------------------------------
counts = pd.crosstab(cc['donor_id'], cc['cc_label'])
N = counts.shape[0]
THRESH = [5, 10, 20, 30, 50]
desc = counts.describe(percentiles=[.1, .5, .9]).T[['min', '10%', '50%', '90%', 'max']]
desc = desc.round(0).astype(int)
desc.insert(0, 'zeros', (counts == 0).sum().values)
print(f"\nper-donor counts on hybrid cc_label ({N} donors):")
print(desc.to_string())
print("\n# donors with >= threshold cells:")
print(pd.DataFrame({f">= {t}": (counts >= t).sum() for t in THRESH}).to_string())

# ---- microglia sparsity vs diagnosis / sex (confound + leakage check) --------
meta = cc.drop_duplicates('donor_id').set_index('donor_id')[['sex', 'condition']]
meta = meta.join(counts['Mic'].rename('mic'))
print(f"\n{'='*70}\nMICROGLIA SPARSITY vs DIAGNOSIS / SEX\n{'='*70}")
print("\nmedian microglia/donor:")
print("  by condition:", meta.groupby('condition', observed=True)['mic'].median().to_dict())
print("  by sex:      ", meta.groupby('sex', observed=True)['mic'].median().to_dict())

for t in (10, 20):
    low = meta['mic'] < t
    print(f"\nlow-microglia (<{t}) x condition:")
    print(pd.crosstab(low, meta['condition']))
    print(f"low-microglia (<{t}) x sex:")
    print(pd.crosstab(low, meta['sex']))

u, p = mannwhitneyu(meta.loc[meta.condition == 'MDD', 'mic'],
                    meta.loc[meta.condition == 'Control', 'mic'])
print(f"\nMann-Whitney microglia/donor MDD vs Control: U={u:.0f}, p={p:.3f}")
print("p>=0.05 => microglia abundance NOT confounded with diagnosis (good).")
print("p<0.05  => abundance differs by diagnosis: note as biological signal AND")
print("           a missingness-leakage path (impute focal edge in-fold only).")

### Step 2 - LIANA per-donor ligand-receptor inference

Consensus L-R inference **per donor** (`sample_key='donor_id'`) on the 8-node
hybrid label, normalized `.X`, `min_cells=10`, `n_perms=None` (magnitude scores
- the right input for Tensor-cell2cell and far faster). Cell types with <10
cells in a donor are masked out for that donor. The long-format result is
checkpointed so this expensive step runs only once.

Runtime: **CPU + High-RAM**. We first subset to L-R-resource genes to cut
memory/time. This is the heavy step - if Colab drags, we move it to the M3.

In [ ]:
# === Phase B - Step 2: LIANA per-donor ligand-receptor inference ===
# Re-run the BOOT cell first (git-pulls functions.py). Needs CPU + High-RAM.
import os, glob, gc, importlib, warnings
warnings.filterwarnings('ignore')          # silence liana/pandas deprecation spam
import numpy as np
import pandas as pd
import scanpy as sc
import liana as li
import functions
importlib.reload(functions)
from functions import make_cc_label

CHECKPOINT_DIR = globals().get('CHECKPOINT_DIR',
    '/content/drive/MyDrive/MLCB_team_project/data/checkpoints')

# --- locate + load the FULL checkpoint (in memory: LIANA needs expression) ----
pats = [os.path.join(CHECKPOINT_DIR, 'phaseA_final_for_cellchat.h5ad'),
        '/content/drive/MyDrive/**/phaseA*cellchat*.h5ad']
CKPT = next((g for pat in pats for g in sorted(glob.glob(pat, recursive=True))), None)
assert CKPT, "checkpoint not found"
print("loading:", CKPT)
adata = sc.read_h5ad(CKPT)                      # full ~5.8 GB load
print(adata)

# --- hybrid 8-node label, drop Mix -------------------------------------------
adata = make_cc_label(adata)                    # adds .obs['cc_label'], drops Mix
print("nodes:", sorted(adata.obs['cc_label'].unique()))
print("donors:", adata.obs['donor_id'].nunique(), "| nuclei:", adata.n_obs)

# --- sanity: .X must be normalized + log1p (Phase A), NOT raw counts ----------
xmax = float(adata.X.max())
print(f".X max = {xmax:.2f}  (log1p-normalized -> single digits; raw counts -> hundreds+)")
assert xmax < 50, "X looks like raw counts; LIANA expects normalized+log1p in .X"

# --- subset to consensus L-R resource genes (memory + speed) -----------------
resource = li.rs.select_resource('consensus')
lr_genes = set()
for col in ('ligand', 'receptor'):
    for entry in resource[col].astype(str).unique():
        lr_genes.update(entry.split('_'))       # split protein complexes
keep = [g for g in adata.var_names if g in lr_genes]
print(f"L-R resource genes present: {len(keep)} / {adata.n_vars}")
adata = adata[:, keep].copy()
gc.collect()

# --- per-DONOR L-R inference (magnitude only) --------------------------------
li.mt.rank_aggregate.by_sample(
    adata,
    sample_key='donor_id',
    groupby='cc_label',
    resource_name='consensus',
    min_cells=10,          # mask cell types with <10 cells in a donor
    expr_prop=0.1,
    use_raw=False,         # .X is normalized + log1p
    n_perms=None,          # magnitude scores only -> fast; right input for tensor
    return_all_lrs=True,
    verbose=True,
)

res = adata.uns['liana_res']
print("\nliana_res shape:", res.shape)
print("columns:", list(res.columns))
sample_col = 'donor_id' if 'donor_id' in res.columns else res.columns[0]
print("sample column:", sample_col, "| donors in result:", res[sample_col].nunique())

# --- checkpoint the per-donor results (never re-run the inference) -----------
out = os.path.join(CHECKPOINT_DIR, 'phaseB_liana_per_donor.parquet')
res.to_parquet(out)
print("saved ->", out)

### Step 3a - Tensor-cell2cell decomposition (fixed rank)

Decompose the 4D communication tensor into communication *programs*. Split for
disconnect-safety: rank selection (slow, one-time) is the OPTIONAL elbow cell
below; **this cell is a single decomposition (~3 min)** at a fixed rank, so
re-running after a disconnect is cheap. Sparse-donor cell types stay masked
(NaN) so the factorization borrows strength across donors. The
**donor x factor** loadings are the Phase C feature set.

In [ ]:
# === Phase B - Step 3a: Tensor-cell2cell decomposition (fixed rank) ===
# Re-run BOOT first. This is the part you re-run after a disconnect: a SINGLE
# decomposition (~3 min), not the 15x elbow. Rank comes from phaseB_rank.json
# if you ran the optional elbow cell, else DEFAULT_RANK.
import os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import liana as li
import cell2cell as c2c

CHECKPOINT_DIR = globals().get('CHECKPOINT_DIR',
    '/content/drive/MyDrive/MLCB_team_project/data/checkpoints')
DEFAULT_RANK = 5   # elbow-selected on the full data

res = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseB_liana_per_donor.parquet'))
print("liana_res:", res.shape, "| donors:", res['donor_id'].nunique())

# --- build the 4D tensor (fast) ----------------------------------------------
tensor = li.multi.to_tensor_c2c(
    liana_res=res, sample_key='donor_id',
    score_key='magnitude_rank', how='outer_cells',
)
print("tensor shape (donor, sender, receiver, LR):", tensor.shape)

# --- rank: from the optional elbow if present, else default ------------------
rank_file = os.path.join(CHECKPOINT_DIR, 'phaseB_rank.json')
have_elbow = os.path.exists(rank_file)
RANK = json.load(open(rank_file))['rank'] if have_elbow else DEFAULT_RANK
print(f"using rank = {RANK}" + (" (from elbow)" if have_elbow else " (default)"))

# --- single decomposition (random init: safe for masked tensors; seeded) -----
tensor.compute_tensor_factorization(rank=RANK, init='random', random_state=0)

# --- save every factor matrix (donor x factor = Phase C features) ------------
factors = tensor.factors
print("factor dims:", list(factors.keys()))
namemap = {'Contexts': 'contexts', 'Sender Cells': 'senders',
           'Receiver Cells': 'receivers', 'Ligand-Receptor Pairs': 'lrs'}
for dim, fac in factors.items():
    short = namemap.get(dim, dim.lower().replace(' ', '_').replace('-', '_'))
    fac.to_parquet(os.path.join(CHECKPOINT_DIR, f'phaseB_factor_{short}.parquet'))
ctx = factors.get('Contexts', list(factors.values())[0])
ctx.to_parquet(os.path.join(CHECKPOINT_DIR, 'phaseB_donor_factors.parquet'))
print("donor x factor loadings:", ctx.shape, "-> saved to", CHECKPOINT_DIR)

### Step 3a (optional) - elbow rank selection, RUN ONCE

The slow part: one decomposition per candidate rank to pick the number of
factors. Rank selection is a **one-time decision** - run this once on a stable
machine (ideally your **M3**, no Colab timeout), or skip it and keep
`DEFAULT_RANK` in Step 3a. When run, it writes `phaseB_rank.json`; re-running
Step 3a then uses it automatically.

In [ ]:
# === Phase B - Step 3a (OPTIONAL): elbow rank selection - RUN ONCE ===
import os, json, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import liana as li
import cell2cell as c2c

CHECKPOINT_DIR = globals().get('CHECKPOINT_DIR',
    '/content/drive/MyDrive/MLCB_team_project/data/checkpoints')

res = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseB_liana_per_donor.parquet'))
tensor = li.multi.to_tensor_c2c(
    liana_res=res, sample_key='donor_id',
    score_key='magnitude_rank', how='outer_cells',
)

# one decomposition per rank 1..upper_rank; automatic_elbow finds the knee
fig, error = tensor.elbow_rank_selection(
    upper_rank=10, runs=1, init='random',
    automatic_elbow=True, random_state=0,
)
print("suggested rank:", tensor.rank)
json.dump({'rank': int(tensor.rank)},
          open(os.path.join(CHECKPOINT_DIR, 'phaseB_rank.json'), 'w'))
print("saved -> phaseB_rank.json  (now re-run Step 3a)")

### Step 3b - Factor interpretation + sex/cohort confound check

Identify which factor is the microglia -> ExN10_L46 communication program
(high `Mic` sender loading x high `ExN10_L46` receiver loading), then run the
mandatory check: does each factor's donor loading track **diagnosis** or
**sex/cohort**? A hypothesis factor should separate case/control - not merely
re-encode the cohort batch.

In [ ]:
# === Phase B - Step 3b: interpret factors + sex/cohort confound check ===
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

CHECKPOINT_DIR = globals().get('CHECKPOINT_DIR',
    '/content/drive/MyDrive/MLCB_team_project/data/checkpoints')

ctx       = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseB_factor_contexts.parquet'))
senders   = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseB_factor_senders.parquet'))
receivers = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseB_factor_receivers.parquet'))
obs       = pd.read_parquet(os.path.join(CHECKPOINT_DIR, 'phaseA_obs.parquet'))
meta = obs.drop_duplicates('donor_id').set_index('donor_id')[['condition', 'sex']]

factor_cols = list(ctx.columns)
print("factors:", factor_cols)

# --- 1. which factor is the microglia -> ExN10_L46 communication program? -----
mic = senders.loc['Mic'].astype(float)
exn = receivers.loc['ExN10_L46'].astype(float)
hypo = (mic * exn).sort_values(ascending=False)     # both endpoints load high
print("\nMic-sender x ExN10_L46-receiver loading product (per factor):")
print(hypo.to_string())
hypo_factor = hypo.index[0]
print(f"\n-> hypothesis factor = {hypo_factor}")
print("   Mic sender %.3f | ExN10_L46 receiver %.3f" % (mic[hypo_factor], exn[hypo_factor]))

# --- 2. per-factor association with diagnosis AND sex/cohort (MANDATORY) ------
df = ctx.join(meta)
rows = []
for f in factor_cols:
    for var, (g0, g1) in [('condition', ('MDD', 'Control')), ('sex', ('female', 'male'))]:
        a = df.loc[df[var] == g0, f]
        b = df.loc[df[var] == g1, f]
        _, p = mannwhitneyu(a, b)
        rows.append({'factor': f, 'var': var, 'median_diff': a.median() - b.median(), 'p': p})
assoc = pd.DataFrame(rows)
print("\nper-factor association (Mann-Whitney p):")
print(assoc.pivot(index='factor', columns='var', values='p').round(4).to_string())

print(f"\n{'='*64}\nCONFOUND READ-OUT - hypothesis factor ({hypo_factor})\n{'='*64}")
sub = assoc[assoc['factor'] == hypo_factor].set_index('var')
print(f"  vs diagnosis:  p = {sub.loc['condition','p']:.4f}")
print(f"  vs sex/cohort: p = {sub.loc['sex','p']:.4f}")
print("Ideal: tracks diagnosis (low p) but NOT sex (high p). If it tracks sex,")
print("the factor carries cohort/batch -> interpret within-sex only.")